In [ ]:
# Load the necessary package
library(dlm)

# Step 1: Define the DLM with trend and seasonal components
# Trend component: Local linear trend (2nd order polynomial)
# Seasonal component: 12 periods (e.g., monthly data)
my_dlm <- dlmModPoly(order = 2, dV = 1, dW = c(0.1, 0.01)) + 
  dlmModSeas(frequency = 12, dV = 0, dW = rep(0.1, 11))
 
# Display the structure of the defined model
cat("Defined DLM structure:\n")
print(my_dlm)

# Step 2: Simulate a non-linear time series with trend and seasonal components
set.seed(123)  # For reproducibility
n <- 500  # Number of time points (e.g., 10 years of monthly data)
time <- 1:n

# Generate a non-linear trend and seasonal pattern
true_trend <- 0.1 * time + sin(time / 20)  # Non-linear trend
true_seasonal <- 5 * cos(2 * pi * time / 12)  # Seasonal component
noise <- rnorm(n, mean = 0, sd = 1)  # Random noise

ts_data <- true_trend + true_seasonal + noise

# Plot the simulated data with high-quality visualization
plot(ts_data, type = "l", col = "blue", lwd = 2,
     main = "Simulated Non-linear Time Series",
     ylab = "Value", xlab = "Time")
lines(true_trend, col = "red", lty = 2, lwd = 2)  # Add the true trend
legend("topright", legend = c("Simulated Data", "True Trend"), 
       col = c("blue", "red"), lty = c(1, 2), bty = "n", cex = 0.8, lwd = c(2, 2))

# Step 3: Fit the DLM to the simulated data
fit <- dlmFilter(ts_data, my_dlm)

# Step 4: Perform smoothing to estimate latent states
smooth_result <- dlmSmooth(fit)

# Extract the smoothed trend and seasonal components
smoothed_trend <- dropFirst(smooth_result$s[, 1])  # Trend (first state component)
smoothed_seasonal <- dropFirst(smooth_result$s[, 3])  # Seasonal (third state component, if modeled explicitly)

# High-quality plot of the results
plot(ts_data, type = "l", col = "blue", lwd = 2,
     main = "Fitted DLM Results",
     ylab = "Value", xlab = "Time")
lines(smoothed_trend, col = "green", lwd = 2)  # Smoothed trend
lines(true_trend, col = "red", lty = 2, lwd = 2)  # True trend
legend("topright", legend = c("Simulated Data", "Smoothed Trend", "True Trend"), 
       col = c("blue", "green", "red"), lty = c(1, 1, 2), bty = "n", cex = 0.8, lwd = c(2, 2))

# Step 5: Evaluate the fit
residuals <- ts_data - dropFirst(fit$m[, 1])  # Observed minus fitted

# High-quality residual plot
plot(residuals, type = "h", col = "darkorange", lwd = 1.5,
     main = "Residuals", ylab = "Residual", xlab = "Time")
abline(h = 0, col = "black", lty = 2)

# Summary statistics for residuals
cat("\nSummary of residuals:\n")
summary(residuals)


Discoiunt factor speficications looks wrong!

In [ ]:
# Load the necessary package
library(dlm)

# Step 1: Define the DLM with trend and seasonal components
# Trend component: Local linear trend (2nd order polynomial)
# Seasonal component: 12 periods (e.g., monthly data)
create_dlm <- function(discount_factor_trend, discount_factor_seasonal, prior) {
  model <- dlmModPoly(order = 2, dV = 1, dW = c(discount_factor_trend, 0.01)) + 
    dlmModSeas(frequency = 12, dV = 0, dW = rep(discount_factor_seasonal, 11))
  
  if (!is.null(prior)) {
    model$m0 <- prior$m0  # Prior mean
    model$C0 <- prior$C0  # Prior covariance matrix
  }
  
  return(model)
}

# Step 2: Simulate a non-linear time series with trend and seasonal components
set.seed(123)  # For reproducibility
n <- 120  # Number of time points (e.g., 10 years of monthly data)
time <- 1:n

# Generate a non-linear trend and seasonal pattern
true_trend <- 0.1 * time + sin(time / 20)  # Non-linear trend
true_seasonal <- 5 * cos(2 * pi * time / 12)  # Seasonal component
noise <- rnorm(n, mean = 0, sd = 1)  # Random noise

ts_data <- true_trend + true_seasonal + noise

# Step 3: Fit the DLM with three discount factors and compare
# Define discount factors and prior
discount_factors <- c(0.1, 1.0)
prior <- list(m0 = c(0, 0), C0 = diag(2) * 10000)  # Example prior

results <- list()

for (d in discount_factors) {
  cat("Fitting DLM with discount factor:", d, "\n")
  dlm_model <- create_dlm(discount_factor_trend = d, discount_factor_seasonal = d, prior = prior)
  fit <- dlmFilter(ts_data, dlm_model)
  smooth_result <- dlmSmooth(fit)
  smoothed_trend <- dropFirst(smooth_result$s[, 1])  # Smoothed trend
  smoothed_seasonal <- dropFirst(smooth_result$s[, 3])  # Smoothed seasonal component
  residuals <- ts_data - dropFirst(fit$m[, 1])  # Residuals
  results[[as.character(d)]] <- list(
    model = dlm_model,
    fit = fit,
    smooth_result = smooth_result,
    smoothed_trend = smoothed_trend,
    smoothed_seasonal = smoothed_seasonal
  )
}

# Step 4: Define a unified ylim for all plots
ylim <- range(c(ts_data, true_trend, true_seasonal), na.rm = TRUE)

# Step 5: Plot the real data and true components
title_main <- "Real Data and True Components"
plot(ts_data, type = "l", col = "blue", lwd = 2, ylim = ylim,
     main = title_main, xlab = "Time", ylab = "Value")
lines(true_trend, col = "red", lwd = 2, lty = 2)
lines(true_seasonal + mean(noise), col = "green", lwd = 2, lty = 2)
legend("topright", legend = c("Real Data", "True Trend", "True Seasonal"),
       col = c("blue", "red", "green"), lty = c(1, 2, 2), lwd = c(2, 2, 2))

# Step 6: Plot fitted models' responses
title_main <- "Fitted Models' Response vs Real Data"
plot(ts_data, type = "l", col = "blue", lwd = 2, ylim = ylim,
     main = title_main, xlab = "Time", ylab = "Value")
for (d in discount_factors) {
  lines(dropFirst(results[[as.character(d)]]$fit$m[, 1]), col = rainbow(length(discount_factors))[which(d == discount_factors)], lwd = 2)
}
legend("topright", legend = c("Real Data", paste("Model: Discount =", discount_factors)),
       col = c("blue", rainbow(length(discount_factors))), lty = 1, lwd = 2)

# Step 7: Plot trend components
title_main <- "Fitted Models' Trend Components vs True Trend"
plot(true_trend, type = "l", col = "red", lwd = 2, ylim = ylim,
     main = title_main, xlab = "Time", ylab = "Trend Value")
for (d in discount_factors) {
  lines(results[[as.character(d)]]$smoothed_trend, col = rainbow(length(discount_factors))[which(d == discount_factors)], lwd = 2)
}
legend("topright", legend = c("True Trend", paste("Model: Discount =", discount_factors)),
       col = c("red", rainbow(length(discount_factors))), lty = 1, lwd = 2)

# Step 8: Plot seasonal components
title_main <- "Fitted Models' Seasonal Components vs True Seasonal"
plot(true_seasonal, type = "l", col = "green", lwd = 2, ylim = ylim,
     main = title_main, xlab = "Time", ylab = "Seasonal Value")
for (d in discount_factors) {
  lines(results[[as.character(d)]]$smoothed_seasonal, col = rainbow(length(discount_factors))[which(d == discount_factors)], lwd = 2)
}
legend("topright", legend = c("True Seasonal", paste("Model: Discount =", discount_factors)),
       col = c("green", rainbow(length(discount_factors))), lty = 1, lwd = 2)

# Step 9: Forecast 60 steps ahead and plot
forecast_horizon <- 60
title_main <- "Forecasts 60 Steps Ahead"
plot(ts_data, type = "l", col = "blue", lwd = 2, ylim = ylim,
     main = title_main, xlab = "Time", ylab = "Value",
     xlim = c(1, n + forecast_horizon))
for (d in discount_factors) {
  forecast <- dlmForecast(results[[as.character(d)]]$fit, nAhead = forecast_horizon)
  forecast_mean <- forecast$f
  forecast_upper <- forecast$f + 2 * sqrt(unlist(forecast$Q))
  forecast_lower <- forecast$f - 2 * sqrt(unlist(forecast$Q))
  lines((n + 1):(n + forecast_horizon), forecast_mean, col = rainbow(length(discount_factors))[which(d == discount_factors)], lwd = 2)
  lines((n + 1):(n + forecast_horizon), forecast_upper, col = rainbow(length(discount_factors))[which(d == discount_factors)], lty = 2)
  lines((n + 1):(n + forecast_horizon), forecast_lower, col = rainbow(length(discount_factors))[which(d == discount_factors)], lty = 2)
}
legend("topright", legend = c("Real Data", paste("Model: Discount =", discount_factors)),
       col = c("blue", rainbow(length(discount_factors))), lty = 1, lwd = 2)
